In [1]:
import pandas as pd
import os
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier

# =======================
# Load and Merge All CSVs
# =======================
input_folder = r"D:\Dataset_Project\dataset_fyp\10Features"

all_data = []
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        print(f"Loading: {filename}")
        df = pd.read_csv(file_path)

        # Drop 'type' if it exists
        if 'type' in df.columns:
            df = df.drop(columns=['type'])

        all_data.append(df)

# Merge into one dataset
df = pd.concat(all_data, axis=0, ignore_index=True)
print(f"\nMerged dataset shape: {df.shape}")

# Split features and labels
y = df['label']
X = df.drop(columns=['label'])

# Ensure binary classification
if len(np.unique(y)) != 2:
    raise ValueError(f"Dataset is not binary. Found {len(np.unique(y))} classes: {np.unique(y)}")

print("\nClass distribution in dataset:")
print(y.value_counts())

# =======================
# Train-test split
# =======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nRunning single train-test split with Tuned XGBoost...\n")

# =======================
# Tuned XGBoost
# =======================
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    n_estimators=1200,       # more boosting rounds
    learning_rate=0.05,      # smaller LR for stability
    max_depth=7,             # deeper trees
    min_child_weight=5,      # regularization to reduce overfitting
    subsample=0.8,           # row sampling
    colsample_bytree=0.8,    # feature sampling
    gamma=0.2,               # min loss reduction (regularization)
    reg_lambda=1.0,          # L2 regularization
    reg_alpha=0.5,           # L1 regularization
    scale_pos_weight=1,      # adjust if dataset is imbalanced
    random_state=42,
    n_jobs=-1
)

# =======================
# Training
# =======================
start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time

# =======================
# Prediction & Metrics
# =======================
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="binary")
recall = recall_score(y_test, y_pred, average="binary")
precision = precision_score(y_test, y_pred, average="binary")

# Confusion matrix & FPR
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
fpr = fp / (fp + tn)

# =======================
# Report Results
# =======================
print("===== Tuned XGBoost Results =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print(f"Training Time: {train_time:.2f} seconds")


Loading: 10Features_preprocess_Network_dataset_14.csv
Loading: 10Features_preprocess_Network_dataset_1.csv
Loading: 10Features_preprocess_Network_dataset_10.csv
Loading: 10Features_preprocess_Network_dataset_11.csv
Loading: 10Features_preprocess_Network_dataset_12.csv
Loading: 10Features_preprocess_Network_dataset_13.csv
Loading: 10Features_preprocess_Network_dataset_15.csv
Loading: 10Features_preprocess_Network_dataset_16.csv
Loading: 10Features_preprocess_Network_dataset_17.csv
Loading: 10Features_preprocess_Network_dataset_18.csv
Loading: 10Features_preprocess_Network_dataset_19.csv
Loading: 10Features_preprocess_Network_dataset_2.csv
Loading: 10Features_preprocess_Network_dataset_20.csv
Loading: 10Features_preprocess_Network_dataset_21.csv
Loading: 10Features_preprocess_Network_dataset_22.csv
Loading: 10Features_preprocess_Network_dataset_23.csv
Loading: 10Features_preprocess_Network_dataset_3.csv
Loading: 10Features_preprocess_Network_dataset_4.csv
Loading: 10Features_preprocess_N

C:\Users\60105\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:183: UserWarning: [22:21:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


===== Tuned XGBoost Results =====
Accuracy: 0.9953
F1 Score: 0.9976
Recall: 0.9994
Precision: 0.9957
False Positive Rate (FPR): 0.1159
Training Time: 1190.90 seconds


In [ ]:
#DT binary
import pandas as pd
import os
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

# Load and Merge All CSVs
input_folder = r"D:\Dataset_Project\dataset_fyp\10Features"

all_data = []
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        print(f"Loading: {filename}")
        df = pd.read_csv(file_path)

        # Drop 'type' if it exists
        if 'type' in df.columns:
            df = df.drop(columns=['type'])

        all_data.append(df)

# Merge into one dataset
df = pd.concat(all_data, axis=0, ignore_index=True)
print(f"\nMerged dataset shape: {df.shape}")

y = df['label']
X = df.drop(columns=['label'])

print("\nClass distribution in dataset:")
print(y.value_counts())

# Train-test split (single run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nRunning single train-test split with Tuned Decision Tree...\n")

# Tuned Decision Tree Parameters
model = DecisionTreeClassifier(
    criterion='entropy',
    class_weight='balanced',
    random_state=10,
    max_depth=11,
    max_leaf_nodes=162,
    min_samples_leaf=20,
    min_impurity_decrease=0.00006
)

start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')

# Confusion matrix (binary only)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
fpr = fp / (fp + tn)

print("\n===== DT Results =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print(f"Training Time: {train_time:.2f} seconds")


Loading: 10Features_preprocess_Network_dataset_14.csv
Loading: 10Features_preprocess_Network_dataset_1.csv
Loading: 10Features_preprocess_Network_dataset_10.csv
Loading: 10Features_preprocess_Network_dataset_11.csv
Loading: 10Features_preprocess_Network_dataset_12.csv
Loading: 10Features_preprocess_Network_dataset_13.csv
Loading: 10Features_preprocess_Network_dataset_15.csv
Loading: 10Features_preprocess_Network_dataset_16.csv
Loading: 10Features_preprocess_Network_dataset_17.csv
Loading: 10Features_preprocess_Network_dataset_18.csv
Loading: 10Features_preprocess_Network_dataset_19.csv
Loading: 10Features_preprocess_Network_dataset_2.csv
Loading: 10Features_preprocess_Network_dataset_20.csv
Loading: 10Features_preprocess_Network_dataset_21.csv
Loading: 10Features_preprocess_Network_dataset_22.csv
Loading: 10Features_preprocess_Network_dataset_23.csv
Loading: 10Features_preprocess_Network_dataset_3.csv
Loading: 10Features_preprocess_Network_dataset_4.csv
Loading: 10Features_preprocess_N

In [ ]:
#GBT binary 
import pandas as pd
import os
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier

# Load and Merge All CSVs
input_folder = r"D:\Dataset_Project\dataset_fyp\10Features"

all_data = []
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        print(f"Loading: {filename}")
        df = pd.read_csv(file_path)

        # Drop 'type' if it exists
        if 'type' in df.columns:
            df = df.drop(columns=['type'])

        all_data.append(df)

# Merge into one dataset
df = pd.concat(all_data, axis=0, ignore_index=True)
print(f"\nMerged dataset shape: {df.shape}")

y = df['label']
X = df.drop(columns=['label'])

print("\nClass distribution in dataset:")
print(y.value_counts())

# Train-test split (single run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nRunning single train-test split with Gradient Boosted Trees...\n")

# Gradient Boosted Trees with tuned parameters
model = GradientBoostingClassifier(
    n_estimators=3200,
    learning_rate=0.05,
    loss='log_loss',
    random_state=42
)

start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
fpr = fp / (fp + tn)

print("\n===== GBT Results =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print(f"Training Time: {train_time:.2f} seconds")


Loading: 10Features_preprocess_Network_dataset_14.csv
Loading: 10Features_preprocess_Network_dataset_1.csv
Loading: 10Features_preprocess_Network_dataset_10.csv
Loading: 10Features_preprocess_Network_dataset_11.csv
Loading: 10Features_preprocess_Network_dataset_12.csv
Loading: 10Features_preprocess_Network_dataset_13.csv
Loading: 10Features_preprocess_Network_dataset_15.csv
Loading: 10Features_preprocess_Network_dataset_16.csv
Loading: 10Features_preprocess_Network_dataset_17.csv
Loading: 10Features_preprocess_Network_dataset_18.csv
Loading: 10Features_preprocess_Network_dataset_19.csv
Loading: 10Features_preprocess_Network_dataset_2.csv
Loading: 10Features_preprocess_Network_dataset_20.csv
Loading: 10Features_preprocess_Network_dataset_21.csv
Loading: 10Features_preprocess_Network_dataset_22.csv
Loading: 10Features_preprocess_Network_dataset_23.csv
Loading: 10Features_preprocess_Network_dataset_3.csv
Loading: 10Features_preprocess_Network_dataset_4.csv
Loading: 10Features_preprocess_N

In [ ]:
#Adaboost binary
import pandas as pd
import os
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Load and Merge All CSVs
input_folder = r"D:\Dataset_Project\dataset_fyp\10Features"

all_data = []
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        print(f"Loading: {filename}")
        df = pd.read_csv(file_path)

        # Drop 'type' if it exists
        if 'type' in df.columns:
            df = df.drop(columns=['type'])

        all_data.append(df)

# Merge into one dataset
df = pd.concat(all_data, axis=0, ignore_index=True)
print(f"\nMerged dataset shape: {df.shape}")

y = df['label']
X = df.drop(columns=['label'])

print("\nClass distribution in dataset:")
print(y.value_counts())

# Ensure binary classification
if len(np.unique(y)) != 2:
    raise ValueError(f"Dataset is not binary. Found {len(np.unique(y))} classes: {np.unique(y)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nRunning single train-test split with AdaBoost (binary only)...\n")

# Base Decision Tree (tuned)
base_dt = DecisionTreeClassifier(
    criterion='gini',
    random_state=10,
    class_weight='balanced',
    max_depth=11,
    max_leaf_nodes=162,
    min_samples_leaf=20,
    min_impurity_decrease=0.00006
)

# AdaBoost (tuned)
model = AdaBoostClassifier(
    estimator=base_dt,
    n_estimators=3300,
    learning_rate=0.3,
    algorithm='SAMME',
    random_state=42
)

# Train
start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time

# Predict
y_pred = model.predict(X_test)

# Binary metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="binary")
recall = recall_score(y_test, y_pred, average="binary")
precision = precision_score(y_test, y_pred, average="binary")

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
fpr = fp / (fp + tn)

# Results
print("\n===== Adaboost Results =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print(f"Training Time: {train_time:.2f} seconds")


Loading: 10Features_preprocess_Network_dataset_14.csv
Loading: 10Features_preprocess_Network_dataset_1.csv
Loading: 10Features_preprocess_Network_dataset_10.csv
Loading: 10Features_preprocess_Network_dataset_11.csv
Loading: 10Features_preprocess_Network_dataset_12.csv
Loading: 10Features_preprocess_Network_dataset_13.csv
Loading: 10Features_preprocess_Network_dataset_15.csv
Loading: 10Features_preprocess_Network_dataset_16.csv
Loading: 10Features_preprocess_Network_dataset_17.csv
Loading: 10Features_preprocess_Network_dataset_18.csv
Loading: 10Features_preprocess_Network_dataset_19.csv
Loading: 10Features_preprocess_Network_dataset_2.csv
Loading: 10Features_preprocess_Network_dataset_20.csv
Loading: 10Features_preprocess_Network_dataset_21.csv
Loading: 10Features_preprocess_Network_dataset_22.csv
Loading: 10Features_preprocess_Network_dataset_23.csv
Loading: 10Features_preprocess_Network_dataset_3.csv
Loading: 10Features_preprocess_Network_dataset_4.csv
Loading: 10Features_preprocess_N

C:\Users\60105\AppData\Roaming\Python\Python312\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(



===== Adaboost Results =====
Accuracy: 0.9789
F1 Score: 0.9890
Recall: 0.9785
Precision: 0.9997
False Positive Rate (FPR): 0.0079
Training Time: 2407.38 seconds


In [9]:
#lgbm default
import pandas as pd
import os
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from lightgbm import LGBMClassifier

# =============================
# Load and Merge All CSVs
# =============================
input_folder = r"D:\Dataset_Project\dataset_fyp\10Features"

all_data = []
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        print(f"Loading: {filename}")
        df = pd.read_csv(file_path)

        # Drop 'type' column if exists
        if 'type' in df.columns:
            df = df.drop(columns=['type'])

        all_data.append(df)

# Merge into one dataset
df = pd.concat(all_data, axis=0, ignore_index=True)
print(f"\nMerged dataset shape: {df.shape}")

y = df['label']
X = df.drop(columns=['label'])

print("\nClass distribution in dataset:")
print(y.value_counts())

# =============================
# Train-test split (80/20)
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nRunning LightGBM (Default Parameters, Binary Only)...\n")

# =============================
# Default LightGBM Model
# =============================
model = LGBMClassifier(random_state=42)

# =============================
# Training
# =============================
start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time

# =============================
# Predictions & Metrics
# =============================
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

# Confusion Matrix & FPR
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)

# =============================
# Results
# =============================
print("\n===== LightGBM Results (Default, Binary) =====")
print(f"Training Time: {train_time:.2f} seconds")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print("Confusion Matrix:")
print(cm)


Loading: 10Features_preprocess_Network_dataset_14.csv
Loading: 10Features_preprocess_Network_dataset_1.csv
Loading: 10Features_preprocess_Network_dataset_10.csv
Loading: 10Features_preprocess_Network_dataset_11.csv
Loading: 10Features_preprocess_Network_dataset_12.csv
Loading: 10Features_preprocess_Network_dataset_13.csv
Loading: 10Features_preprocess_Network_dataset_15.csv
Loading: 10Features_preprocess_Network_dataset_16.csv
Loading: 10Features_preprocess_Network_dataset_17.csv
Loading: 10Features_preprocess_Network_dataset_18.csv
Loading: 10Features_preprocess_Network_dataset_19.csv
Loading: 10Features_preprocess_Network_dataset_2.csv
Loading: 10Features_preprocess_Network_dataset_20.csv
Loading: 10Features_preprocess_Network_dataset_21.csv
Loading: 10Features_preprocess_Network_dataset_22.csv
Loading: 10Features_preprocess_Network_dataset_23.csv
Loading: 10Features_preprocess_Network_dataset_3.csv
Loading: 10Features_preprocess_Network_dataset_4.csv
Loading: 10Features_preprocess_N